# Document Loaders in LangChain

LangChain represents any piece of loaded content as a `Document`: a `page_content` string plus a `metadata` dict (e.g. `source`, `page`, `row`). A **document loader** is just something that reads a file (or files) and produces a list of `Document` objects. This notebook walks through loading the file types found in `data/`: PDF (single + multiple), CSV, TXT, and MD.

All loaders below currently live in `langchain_community.document_loaders` — that's still the correct, documented import path today, though `langchain-community` itself is marked as being gradually sunset in favor of standalone integration packages, so expect import paths for individual loaders to keep migrating over time.


## 1. Loading a single PDF — `PyPDFLoader`

`PyPDFLoader` (from `langchain_community`, backed by the `pypdf` library) returns one `Document` per page.

In [2]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader("jesc1ps.pdf")
docs = loader.load()

print(f"Loaded {len(docs)} page(s)")
print(docs[0].page_content[:300])
print(docs[0].metadata)

C:\Users\hp\AppData\Local\Temp\ipykernel_10280\119288189.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


Loaded 2 page(s)
�������
TEXTBOOK FOR CLASS X
Reprint 2026-27
{'producer': 'cairo 1.18.0 (https://cairographics.org)', 'creator': 'PyPDF', 'creationdate': '2026-08-03T19:30:14+05:30', 'source': 'jesc1ps.pdf', 'total_pages': 2, 'page': 0, 'page_label': '1'}


## 2. A faster PDF loader — `PyMuPDFLoader`

`PyMuPDFLoader` (backed by the `pymupdf` library) parses PDFs faster than `PyPDFLoader`, which matters once you're loading many or large PDFs. The gap barely shows on a tiny 7-page file, so this compares both loaders on the 296-page `SCHEMES.pdf`. (`pymupdf`'s own one-time import cost is excluded by importing it before timing starts.)

In [3]:
import time

from langchain_community.document_loaders import PyMuPDFLoader  # import first so the timing below is fair

BIG_PDF = "data/SCHEMES.pdf"

start = time.perf_counter()
pypdf_big_docs = PyPDFLoader(BIG_PDF).load()
pypdf_time = time.perf_counter() - start

start = time.perf_counter()
pymupdf_big_docs = PyMuPDFLoader(BIG_PDF).load()
pymupdf_time = time.perf_counter() - start

print(f"PyPDFLoader:   {pypdf_time:.4f}s for {len(pypdf_big_docs)} pages")
print(f"PyMuPDFLoader: {pymupdf_time:.4f}s for {len(pymupdf_big_docs)} pages")

PyPDFLoader:   8.6037s for 296 pages
PyMuPDFLoader: 0.5925s for 296 pages


## 3. Loading multiple PDFs at once — `DirectoryLoader`

`DirectoryLoader` batch-loads every file matching a glob pattern in a folder, delegating each file to the loader class you give it via `loader_cls`. `data/` has three PDFs — this loads all of them in one call.

In [4]:
from langchain_community.document_loaders import DirectoryLoader

pdf_loader = DirectoryLoader(
    "data",
    glob="*.pdf",
    loader_cls=PyMuPDFLoader,
)
pdf_docs = pdf_loader.load()

sources = {}
for doc in pdf_docs:
    sources[doc.metadata["source"]] = sources.get(doc.metadata["source"], 0) + 1

print(f"Loaded {len(pdf_docs)} page(s) total from {len(sources)} PDF file(s)")
for source, count in sources.items():
    print(f"  {source}: {count} page(s)")

Loaded 312 page(s) total from 3 PDF file(s)
  data\ai_summit.pdf: 7 page(s)
  data\sampled_schemes.pdf: 9 page(s)
  data\SCHEMES.pdf: 296 page(s)


## 4. Loading a CSV — `CSVLoader`

`CSVLoader` turns each row into one `Document`. `page_content` is the row rendered as `column: value` lines, and `metadata` tracks the source file and row index.

In [5]:
from langchain_community.document_loaders import CSVLoader

loader = CSVLoader("data/sample_products.csv")
csv_docs = loader.load()

print(f"Loaded {len(csv_docs)} row(s)")
print(csv_docs[0].page_content)
print(csv_docs[0].metadata)

Loaded 4 row(s)
Product: Laptop
Category: Electronics
Price: $999
Stock: 25
{'source': 'data/sample_products.csv', 'row': 0}


## 5. Loading TXT files — `TextLoader`

`TextLoader` reads a plain text file into a single `Document`. Combine it with `DirectoryLoader` to load every `.txt` file in `data/` at once.

In [6]:
from langchain_community.document_loaders import TextLoader

loader = TextLoader("data/dc.txt")
txt_docs = loader.load()

print(txt_docs[0].page_content[:300])
print(txt_docs[0].metadata)

A **Document Loader** is a component (commonly used in LLM / RAG systems) that **reads documents from different sources and converts them into structured text format** so they can be processed by models.

Since youâ€™re working on RAG systems and legal document processing, this is especially importa
{'source': 'data/dc.txt'}


In [7]:
txt_loader = DirectoryLoader(
    "data",
    glob="*.txt",
    loader_cls=TextLoader,
    loader_kwargs={"encoding": "utf-8"},
)
all_txt_docs = txt_loader.load()

print(f"Loaded {len(all_txt_docs)} txt file(s)")
for doc in all_txt_docs:
    print(f"  {doc.metadata['source']}: {print(doc.page_content)} characters")

Loaded 3 txt file(s)
A **Document Loader** is a component (commonly used in LLM / RAG systems) that **reads documents from different sources and converts them into structured text format** so they can be processed by models.

Since you’re working on RAG systems and legal document processing, this is especially important for you.

---

## 🔹 What Does a Document Loader Do?

A document loader:

1. **Reads files** (PDF, DOCX, TXT, JSON, CSV, HTML, etc.)
2. **Extracts text content**
3. **Converts it into a standard format**
4. **Creates document objects with metadata**
5. Passes them to:

   * Text splitters
   * Embedding models
   * Vector databases

---

## 🔹 Why It Is Important in RAG (Retrieval Augmented Generation)

In a RAG pipeline:

```
Raw Files → Document Loader → Text Splitter → Embeddings → Vector DB → Retriever → LLM
```

Without a document loader:

* Your model cannot read PDFs
* Cannot extract text from DOCX
* Cannot handle structured JSON properly
* Cannot attach metadata (

## 6. Loading Markdown — `TextLoader`

Markdown is plain text, so `TextLoader` loads it just like a `.txt` file — `page_content` keeps the raw `#`/`*`/`>` syntax as-is. If you need the headings, lists, and tables parsed into structured elements instead of raw text, use `UnstructuredMarkdownLoader` (heavier dependency: `pip install unstructured markdown`).

In [8]:
loader = TextLoader("data/notes.md")
md_docs = loader.load()

print(md_docs[0].page_content)
print(md_docs[0].metadata)

# Structure-aware alternative (splits headings/lists/tables into elements):
#
#   pip install unstructured markdown
#
# from langchain_community.document_loaders import UnstructuredMarkdownLoader
# loader = UnstructuredMarkdownLoader("data/notes.md", mode="elements")
# md_element_docs = loader.load()

# RAG Pipeline Notes

Quick reference notes on building a retrieval-augmented generation pipeline.

## Stages

1. **Load** â€” read raw files (PDF, CSV, TXT, MD) into `Document` objects.
2. **Split** â€” chunk long documents so they fit an embedding model's context window.
3. **Embed** â€” turn each chunk into a vector.
4. **Retrieve** â€” find the chunks most similar to a query.
5. **Generate** â€” pass retrieved chunks to an LLM as context.

## Things to watch for

- Chunk size and overlap change retrieval quality a lot.
- Always keep `source` metadata so answers can be traced back to a file.
- CSV rows and PDF pages need different chunking strategies than free text.

> A document loader's only job is step 1 â€” everything downstream assumes it produced clean, well-tagged `Document` objects.

{'source': 'data/notes.md'}


In [9]:

from langchain_community.document_loaders import UnstructuredMarkdownLoader
loader = UnstructuredMarkdownLoader("data/notes.md", mode="elements")
md_element_docs = loader.load()

print(md_element_docs[0].page_content)
print(md_element_docs[0].metadata)

RAG Pipeline Notes
{'source': 'data/notes.md', 'category_depth': 0, 'languages': ['eng'], 'file_directory': 'data', 'filename': 'notes.md', 'filetype': 'text/markdown', 'last_modified': '2026-08-15T17:55:03', 'category': 'Title', 'element_id': '0f417d7edcacf606130bcadc0db1e2db'}


## 7. Putting it together

Every loader above returns the same thing: a list of `Document` objects. That means they can be freely combined, then split with a text splitter (like in the RAG notebook) before embedding.

In [14]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

all_docs = pdf_docs + csv_docs + all_txt_docs + md_docs
print(f"{len(all_docs)} documents loaded from data/ across PDF, CSV, TXT, and MD")

splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
chunks = splitter.split_documents(all_docs)
print(f"Split into {len(chunks)} chunks")
print(chunks[140].page_content[:200])


320 documents loaded from data/ across PDF, CSV, TXT, and MD
Split into 1343 chunks
15 
 
provided to cover yield losses due to non-preventable risks, viz. Drought, 
Dry spell, Flood, Inundation, widespread Pests and Disease attack, 
Landslides, Fire due to natural causes, Lightening


## Summary

| Loader | File type | Package needed |
|---|---|---|
| `PyPDFLoader` | `.pdf` | `langchain-community`, `pypdf` |
| `PyMuPDFLoader` | `.pdf` (faster) | `langchain-community`, `pymupdf` |
| `DirectoryLoader` | many files of one type at once | `langchain-community` |
| `CSVLoader` | `.csv` | `langchain-community` |
| `TextLoader` | `.txt`, `.md` | `langchain-community` |
| `UnstructuredMarkdownLoader` | `.md` (structure-aware) | `langchain-community`, `unstructured`, `markdown` |

